# 12 — Bancada de testes (4 cabeças): rodar o modelo em qualquer documento, sem gabarito

Diferente dos notebooks 10/11 (que exigem o gabarito de holdout alinhado, notebooks 07/09 em `notebooks/bert/`), este notebook serve para testar rapidamente **qualquer** documento — um PDF avulso ou um JSON já extraído — e só olhar o que o modelo prevê, sem comparar com nada. Feito para explorar casos pontuais (ex.: reproduzir/testar o caso do "segundo endereço na mesma frase", validar uma ideia de pós-processamento) sem rodar o pipeline de alinhamento inteiro.

**Documentos de teste avulsos ficam em `data/documento_social/teste_geral/`**, espelhando a estrutura de `teste_holdout/`: coloque o arquivo original em `teste_geral/arquivos/`, e o JSON de extração (quando precisar de OCR) vai para `teste_geral/extracao/`.

**Duas formas de apontar o documento:**
1. `CAMINHO_PDF` — um PDF com **texto nativo** (não escaneado). Este notebook extrai o texto direto via PyMuPDF, sem precisar do kernel `locscan-ocr`.
2. `CAMINHO_JSON_EXTRACAO` + `CAMINHO_PDF_VISUALIZACAO` — para documento escaneado **ou imagem** (imagem nunca tem texto nativo, sempre precisa desse caminho): rode antes `notebooks/bert/02b.extrair_um_arquivo_avulso.ipynb` (kernel `locscan-ocr`) sobre o arquivo em `teste_geral/arquivos/`, e aponte aqui para o JSON gerado em `teste_geral/extracao/` e para o arquivo original (só para desenhar as caixas na visualização — PyMuPDF abre imagem normalmente também, não precisa ser PDF).

Carrega o modelo treinado em `notebooks/bert_4_cabecas/06.treinar_bertimbau_4_cabecas.ipynb` (encoder compartilhado + 4 cabeças de classificação). Para a variante de modelo único, ver `notebooks/bert/12...ipynb`.

**Pré-requisito:** venv GERAL do projeto (`.venv`, torch/transformers) — nunca o `.venv_ocr`.

In [ ]:
import json
import sys
from pathlib import Path

import fitz  # PyMuPDF
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import torch
import torch.nn as nn
from PIL import Image
from io import BytesIO
from transformers import AutoModel, AutoTokenizer

print(f"CUDA disponível: {torch.cuda.is_available()}")
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Configuração

In [ ]:
def encontrar_raiz_projeto(marcador: str = "requirements.txt") -> Path:
    p = Path.cwd()
    for _ in range(6):
        if (p / marcador).exists():
            return p
        p = p.parent
    raise RuntimeError(f"Não encontrei a raiz do projeto ({marcador}) subindo a partir de {Path.cwd()}")

RAIZ = encontrar_raiz_projeto()
sys.path.insert(0, str(RAIZ / "scripts"))
from pos_processamento_ner import extrair_nome_assinatura_digital, propagar_entidades_por_texto_identico

PASTA_TESTE_GERAL = RAIZ / "data" / "documento_social" / "teste_geral"

# --- Escolha do documento (preencha UMA das duas opções) ---
# Imagem (.jpg/.png/...) SEMPRE precisa passar pelo notebook 02b (OCR) antes -- não tem texto
# nativo pra extrair direto. Ex.:
#   CAMINHO_PDF = None
#   CAMINHO_JSON_EXTRACAO = PASTA_TESTE_GERAL / "extracao" / "algum.json"
#   CAMINHO_PDF_VISUALIZACAO = PASTA_TESTE_GERAL / "arquivos" / "algum.png"
CAMINHO_PDF = None
CAMINHO_JSON_EXTRACAO = None
CAMINHO_PDF_VISUALIZACAO = None

# --- Escolha de experimento/cenário ---
INCLUIR_SINTETICOS = False
CENARIO_MANUAL = None   # ex: "c08_warmup" -- None usa o melhor por F1 do experimento

APLICAR_REGEX_ASSINATURA_DIGITAL = True
APLICAR_PROPAGACAO_TEXTO_IDENTICO = True

MAX_LENGTH = 512
TIPOS = ["CPF", "RG", "NOME", "ENDERECO"]

_sufixo_experimento = "-com-sinteticos" if INCLUIR_SINTETICOS else "-sem-sinteticos"
PASTA_CENARIOS = RAIZ / "models" / f"bertimbau-4cabecas-pii-v1-cenarios{_sufixo_experimento}"

resultados_cenarios = json.loads((PASTA_CENARIOS / "_comparacao_cenarios.json").read_text(encoding="utf-8"))
cenario_escolhido = next((r for r in resultados_cenarios if r["nome"] == CENARIO_MANUAL), resultados_cenarios[0]) if CENARIO_MANUAL else resultados_cenarios[0]

print(f"Experimento: {PASTA_CENARIOS.name}")
print(f"Cenário: {cenario_escolhido['nome']} (F1 token-level={cenario_escolhido['eval_f1']:.4f})")
print(f"Pasta do modelo: {cenario_escolhido['pasta_modelo']}")

## Carrega o documento (PDF nativo ou JSON já extraído)

Se `CAMINHO_PDF` tiver texto nativo (a maioria dos documentos societários tem, mesmo quando algumas páginas são scan), a extração roda direto aqui. Páginas sem texto extraível (ex.: scan puro, ou imagem) ficam com 0 tokens e são avisadas -- para essas, é preciso rodar o notebook 02b (em `notebooks/bert/`, kernel `locscan-ocr`) e usar `CAMINHO_JSON_EXTRACAO` em vez de `CAMINHO_PDF`.

In [ ]:
def extrair_nativo(caminho_pdf: Path) -> dict:
    doc = fitz.open(caminho_pdf)
    paginas = []
    paginas_vazias = []
    for i, pagina in enumerate(doc, start=1):
        palavras = pagina.get_text("words")
        tokens = [{"text": w[4], "bbox": [w[0], w[1], w[2], w[3]]} for w in palavras]
        if not tokens:
            paginas_vazias.append(i)
        paginas.append({"page": i, "tokens": tokens})
    doc.close()
    if paginas_vazias:
        print(
            f"[AVISO] {len(paginas_vazias)} página(s) sem texto nativo extraível: {paginas_vazias} -- "
            f"provavelmente scan puro. Rode notebooks/bert/02 (kernel locscan-ocr) e use "
            f"CAMINHO_JSON_EXTRACAO em vez de CAMINHO_PDF para essas páginas."
        )
    return {"document_id": Path(caminho_pdf).name, "pages": paginas}


if CAMINHO_PDF is not None:
    extracao = extrair_nativo(Path(CAMINHO_PDF))
    caminho_pdf_visualizacao = Path(CAMINHO_PDF)
elif CAMINHO_JSON_EXTRACAO is not None:
    extracao = json.loads(Path(CAMINHO_JSON_EXTRACAO).read_text(encoding="utf-8"))
    if CAMINHO_PDF_VISUALIZACAO is None:
        raise ValueError("Defina CAMINHO_PDF_VISUALIZACAO junto com CAMINHO_JSON_EXTRACAO (para desenhar as caixas).")
    caminho_pdf_visualizacao = Path(CAMINHO_PDF_VISUALIZACAO)
else:
    raise ValueError("Preencha CAMINHO_PDF ou (CAMINHO_JSON_EXTRACAO + CAMINHO_PDF_VISUALIZACAO) na célula de configuração.")

n_paginas = len(extracao["pages"])
n_tokens = sum(len(p["tokens"]) for p in extracao["pages"])
print(f"Documento: {extracao['document_id']} -- {n_paginas} página(s), {n_tokens} token(s) no total.")

## Carrega o modelo (encoder + 4 cabeças) e roda a inferência

In [ ]:
class BERTimbau4Cabecas(nn.Module):
    """Mesma definição de notebooks/bert_4_cabecas/06 -- reproduzida aqui por autocontenção."""

    def __init__(self, checkpoint_base: str, tipos: list):
        super().__init__()
        self.tipos = list(tipos)
        self.encoder = AutoModel.from_pretrained(checkpoint_base)
        hidden = self.encoder.config.hidden_size
        p_dropout = getattr(self.encoder.config, "hidden_dropout_prob", 0.1)
        self.dropout = nn.Dropout(p_dropout)
        self.cabecas = nn.ModuleDict({tipo: nn.Linear(hidden, 3) for tipo in self.tipos})

    def forward(self, input_ids, attention_mask):
        saida_encoder = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        oculto = self.dropout(saida_encoder.last_hidden_state)
        return {"logits_por_tipo": {tipo: self.cabecas[tipo](oculto) for tipo in self.tipos}}

    @classmethod
    def load(cls, pasta, device="cpu"):
        pasta = Path(pasta)
        config = json.loads((pasta / "config_4cabecas.json").read_text(encoding="utf-8"))
        modelo = cls(str(pasta / "encoder"), config["tipos"])
        cabecas_state = torch.load(pasta / "cabecas.pt", map_location=device)
        for tipo in config["tipos"]:
            modelo.cabecas[tipo].load_state_dict(cabecas_state[tipo])
        return modelo.to(device)


LABEL2ID_POR_TIPO = {tipo: {"O": 0, f"B-{tipo}": 1, f"I-{tipo}": 2} for tipo in TIPOS}
ID2LABEL_POR_TIPO = {tipo: {v: k for k, v in LABEL2ID_POR_TIPO[tipo].items()} for tipo in TIPOS}

tokenizer = AutoTokenizer.from_pretrained(Path(cenario_escolhido["pasta_modelo"]) / "encoder")
model = BERTimbau4Cabecas.load(cenario_escolhido["pasta_modelo"], device=device)
model.eval()
print(f"Modelo carregado: {cenario_escolhido['nome']}")


def agrupar_bboxes_por_linha(tokens_bbox: list, indices: list) -> list:
    if not indices:
        return []
    grupos = []
    grupo_atual = [tokens_bbox[indices[0]]]
    y_ref = tokens_bbox[indices[0]][1]
    for i in indices[1:]:
        bbox = tokens_bbox[i]
        altura_ref = grupo_atual[-1][3] - grupo_atual[-1][1]
        if abs(bbox[1] - y_ref) > max(altura_ref * 0.6, 3):
            grupos.append(grupo_atual)
            grupo_atual = [bbox]
            y_ref = bbox[1]
        else:
            grupo_atual.append(bbox)
    grupos.append(grupo_atual)
    return [[min(b[0] for b in g), min(b[1] for b in g), max(b[2] for b in g), max(b[3] for b in g)] for g in grupos]


from pos_processamento_ner import mesclar_previsoes_por_janela


@torch.no_grad()
def _prever_tags_janela_tipo(tokens_janela: list, tipo: str) -> list:
    encoding = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH, return_tensors="pt")
    encoding = {k: v.to(device) for k, v in encoding.items()}
    saida = model(**encoding)
    pred_ids = saida["logits_por_tipo"][tipo][0].argmax(dim=-1).tolist()
    word_ids = tokenizer(tokens_janela, is_split_into_words=True, truncation=True, max_length=MAX_LENGTH).word_ids()
    tags_por_palavra = ["O"] * len(tokens_janela)
    visto = set()
    for pid, word_idx in zip(pred_ids, word_ids):
        if word_idx is None or word_idx in visto:
            continue
        visto.add(word_idx)
        tags_por_palavra[word_idx] = ID2LABEL_POR_TIPO[tipo][pid]
    return tags_por_palavra


def prever_pagina(tokens: list, tokens_bbox: list) -> list:
    """Prevê a página/imagem inteira, uma cabeça (tipo) por vez, dividindo em janelas
    deslizantes com sobreposição quando ultrapassa MAX_LENGTH subtokens (achado 2026-09-15,
    caso MINAS CAL/segundo endereço truncado -- ver scripts/pos_processamento_ner.py)."""
    if not tokens:
        return []
    entidades = []
    for tipo in TIPOS:
        tags_por_palavra = mesclar_previsoes_por_janela(
            tokens, tokenizer, lambda tj, t=tipo: _prever_tags_janela_tipo(tj, t), max_length=MAX_LENGTH
        )
        i, n = 0, len(tags_por_palavra)
        while i < n:
            if tags_por_palavra[i] == "O":
                i += 1
                continue
            indices = [i]
            j = i + 1
            while j < n and tags_por_palavra[j] == f"I-{tipo}":
                indices.append(j)
                j += 1
            entidades.append({
                "type": tipo,
                "bbox": agrupar_bboxes_por_linha(tokens_bbox, indices),
                "indices": indices,
                "text": " ".join(tokens[k] for k in indices),
            })
            i = j
    return entidades


def bbox_externo(lista_bboxes: list):
    if not lista_bboxes:
        return None
    return [min(b[0] for b in lista_bboxes), min(b[1] for b in lista_bboxes),
            max(b[2] for b in lista_bboxes), max(b[3] for b in lista_bboxes)]


def iou(a, b) -> float:
    if a is None or b is None:
        return 0.0
    xi0, yi0 = max(a[0], b[0]), max(a[1], b[1])
    xi1, yi1 = min(a[2], b[2]), min(a[3], b[3])
    inter = max(0, xi1 - xi0) * max(0, yi1 - yi0)
    if inter == 0:
        return 0.0
    area_a = max(0, a[2] - a[0]) * max(0, a[3] - a[1])
    area_b = max(0, b[2] - b[0]) * max(0, b[3] - b[1])
    uniao = area_a + area_b - inter
    return inter / uniao if uniao > 0 else 0.0

## Roda a inferência (sem gabarito) — texto puro, um print por entidade encontrada

Pós-processamento aplicado se as flags `APLICAR_REGEX_ASSINATURA_DIGITAL`/`APLICAR_PROPAGACAO_TEXTO_IDENTICO` estiverem ligadas -- mesmas funções de `scripts/pos_processamento_ner.py` (arquitetura-agnósticas).

In [ ]:
entidades_documento = []

for pagina_dados in extracao["pages"]:
    tokens_dict = pagina_dados["tokens"]
    if not tokens_dict:
        continue
    tokens = [t["text"] for t in tokens_dict]
    tokens_bbox = [t["bbox"] for t in tokens_dict]

    previstas = prever_pagina(tokens, tokens_bbox)

    if APLICAR_REGEX_ASSINATURA_DIGITAL:
        extras = extrair_nome_assinatura_digital(tokens, tokens_bbox)
        for extra in extras:
            bbox_extra = bbox_externo(extra["bbox"])
            ja_coberto = any(
                p["type"] == "NOME" and iou(bbox_extra, bbox_externo(p["bbox"])) > 0.1
                for p in previstas
            )
            if not ja_coberto:
                previstas.append(extra)

    for p in previstas:
        p["page"] = pagina_dados["page"]
    entidades_documento.extend(previstas)

if APLICAR_PROPAGACAO_TEXTO_IDENTICO:
    entidades_documento = propagar_entidades_por_texto_identico(
        entidades_documento, extracao["pages"], tipos_propagaveis=tuple(TIPOS)
    )

print(f"{extracao['document_id']} -- {len(entidades_documento)} entidade(s) encontrada(s)\n")
for e in sorted(entidades_documento, key=lambda x: (x["page"], x["type"])):
    origem = f" [{e['origem']}]" if "origem" in e else ""
    print(f"  p{e['page']:<3} {e['type']:<9} \"{e['text']}\"{origem}")

## Visualização: todas as páginas com as caixas previstas

Preencha `PAGINA_UNICA` com um número para ver só uma página, ou deixe `None` para ver o documento inteiro (uma figura por página).

In [ ]:
CORES_POR_TIPO = {
    "CPF": (0.85, 0.1, 0.1),
    "RG": (0.1, 0.4, 0.85),
    "NOME": (0.1, 0.7, 0.2),
    "ENDERECO": (0.8, 0.6, 0.0),
}
PAGINA_UNICA = None  # ex: 3 -- None mostra todas as páginas
DPI_VISUALIZACAO = 150

EXTENSOES_IMAGEM_VIS = {".jpg", ".jpeg", ".png", ".tif", ".tiff", ".bmp"}
legenda = "  ".join(f"{t}={c}" for t, c in CORES_POR_TIPO.items())


def desenhar_pagina(imagem_base, entidades_pagina, titulo):
    # Desenha os bbox por cima da imagem via matplotlib (em vez de page.draw_rect, que só
    # funciona em documento PDF de verdade -- Shape.__init__ do PyMuPDF recusa imagem com
    # "ValueError: is no PDF"). Funciona igual para PDF renderizado e para imagem direta.
    fig, ax = plt.subplots(figsize=(11, 15))
    ax.imshow(imagem_base)
    for e in entidades_pagina:
        cor = CORES_POR_TIPO.get(e["type"], (0, 0, 0))
        # Um retângulo por linha de e["bbox"] -- bbox_externo() (um retângulo único
        # cobrindo todas as linhas) faz o desenho "vazar" para a esquerda quando a
        # entidade quebra linha e a 2ª linha começa mais à esquerda que a 1ª (achado
        # 2026-09-15, caso MINAS CAL/endereço do Sérgio).
        for retangulo in (e["bbox"] or []):
            x0, y0, x1, y1 = retangulo
            ax.add_patch(patches.Rectangle((x0, y0), x1 - x0, y1 - y0, linewidth=1.5, edgecolor=cor, facecolor="none"))
    ax.axis("off")
    ax.set_title(f"{titulo}\n{legenda}")
    plt.tight_layout()
    plt.show()


if Path(caminho_pdf_visualizacao).suffix.lower() in EXTENSOES_IMAGEM_VIS:
    # Imagem: o bbox já está no espaço de pixel nativo da própria imagem (ver notebook 02b,
    # processar_imagem -- sem escala de DPI aplicada), desenha direto sobre ela, sem passar por fitz.
    imagem_base = Image.open(caminho_pdf_visualizacao)
    entidades_pagina = [e for e in entidades_documento if e["page"] == 1]
    desenhar_pagina(
        imagem_base, entidades_pagina,
        f"{extracao['document_id']} ({len(entidades_pagina)} entidade(s))",
    )
else:
    doc_pdf = fitz.open(caminho_pdf_visualizacao)
    escala_pdf_para_pixel = DPI_VISUALIZACAO / 72.0
    paginas_para_mostrar = [PAGINA_UNICA] if PAGINA_UNICA else range(1, doc_pdf.page_count + 1)

    for numero_pagina in paginas_para_mostrar:
        pagina_pdf = doc_pdf[numero_pagina - 1]
        entidades_pagina = [e for e in entidades_documento if e["page"] == numero_pagina]

        pix = pagina_pdf.get_pixmap(dpi=DPI_VISUALIZACAO)
        imagem_base = Image.open(BytesIO(pix.tobytes("png")))

        # bbox está em pontos PDF (referência 72 dpi) -- converte pra pixel na resolução renderizada
        entidades_pagina_px = [
            {**e, "bbox": [[c * escala_pdf_para_pixel for c in caixa] for caixa in e["bbox"]]}
            for e in entidades_pagina
        ]
        desenhar_pagina(
            imagem_base, entidades_pagina_px,
            f"{extracao['document_id']} — página {numero_pagina}/{doc_pdf.page_count} ({len(entidades_pagina)} entidade(s))",
        )
    doc_pdf.close()